<a href="https://colab.research.google.com/github/Mdsinan09/colab-research/blob/main/Travel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ⚠️  IMPORTANT — Read before running
# After this cell finishes, go to:  Runtime → Restart Runtime
# Then run all cells again from the top (Runtime → Run All)
# This is required in Google Colab whenever new packages are installed.

!pip install -q -U \
    "langchain>=0.3.7,<0.4.0" \
    "langchain-groq>=0.2.0" \
    "langchain-community>=0.3.7" \
    "langchain-core>=0.3.7" \
    duckduckgo-search \
    requests

# Verify installed versions
import langchain, langchain_groq, langchain_community
print(f'✅ langchain          : {langchain.__version__}')
print(f'✅ langchain-groq     : {langchain_groq.__version__}')
print(f'✅ langchain-community: {langchain_community.__version__}')
print()
print('⚠️  NOW: Runtime → Restart Runtime → then run all cells again.')

✅ langchain          : 0.3.30
✅ langchain-groq     : 0.3.8
✅ langchain-community: 0.3.31

⚠️  NOW: Runtime → Restart Runtime → then run all cells again.


In [ ]:
import os, requests, getpass

# LangChain — Core
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

# LangChain — Groq LLM
from langchain_groq import ChatGroq

# NOTE: We intentionally do NOT use AgentExecutor.
# We write a simple manual agent loop instead — more reliable with Groq
# and more educational (you see exactly how the loop works).

# LangChain — Memory (for Session 2)
from langchain_community.chat_message_histories import ChatMessageHistory

# DuckDuckGo Search
from duckduckgo_search import DDGS

import langchain
print(f'✅ All libraries imported  |  LangChain {langchain.__version__}')

✅ All libraries imported  |  LangChain 0.3.30


In [ ]:
# Securely enter your API keys — not stored, session-only
GROQ_API_KEY = getpass.getpass('🔑 Enter your Groq API Key: ')
OPENWEATHER_API_KEY = getpass.getpass('🌤️  Enter your OpenWeatherMap API Key: ')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

print('✅ API keys configured!')

🔑 Enter your Groq API Key: ··········
🌤️  Enter your OpenWeatherMap API Key: ··········
✅ API keys configured!


In [ ]:
# ── MODEL CHOICE ────────────────────────────────────────────
GROQ_MODEL = 'qwen/qwen3.6-27b'   # smaller, faster, same tool support
    # GROQ_MODEL = 'llama-3.3-70b-versatile'
llm = ChatGroq(
        model=GROQ_MODEL,
        temperature=0,
        max_tokens=2048,
        streaming=False,      # must be False for reliable tool calling
        api_key=GROQ_API_KEY
    )
response = llm.invoke('Give one travel fact about India in one sentence.')
print(f'✅ LLM ready — model: {GROQ_MODEL}')
print('🤖', response.content)


✅ LLM ready — model: qwen/qwen3.6-27b
🤖 
<think>
Thinking Process:

1.  **Deconstruct the user's request:**
    *   Topic: Travel fact about India.
    *   Quantity: One fact.
    *   Format: One sentence.

2.  **Brainstorm potential facts:**
    *   India has the Taj Mahal. (Too cliché?)
    *   India has the world's largest democracy. (Political, not strictly travel-focused, though related.)
    *   India has the highest railway crossing in the world. (Good, specific.)
    *   India has the longest railway platform. (Good, specific.)
    *   India has the only place where you can see the sun rise and set from the same spot? (No, that's not unique or accurate.)
    *   India has the world's largest animal migration? (No, that's usually Serengeti, though India has bird migrations.)
    *   India has the only place where you can see the Himalayas and the Indian Ocean? (Kerala? Maybe, but not a "fact" per se.)
    *   India has the world's largest postal network? (Interesting, but maybe 

In [ ]:
# ── BUILD YOUR FIRST CHAIN ──────────────────────────────────

# Step A: Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful and enthusiastic travel assistant.'),
    ('human', '{input}')
])

# Step B: Output Parser — converts AIMessage object to plain string
parser = StrOutputParser()

# Step C: Assemble the Chain using the pipe operator
chain = prompt | llm | parser

print('✅ Chain built: prompt | llm | parser')

✅ Chain built: prompt | llm | parser


In [ ]:
# ── RUN THE CHAIN ───────────────────────────────────────────

response = chain.invoke({'input': 'What are the top 3 things to do in Kerala?'})
print('🤖 Travel Assistant:')
print(response)

🤖 Travel Assistant:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "What are the top 3 things to do in Kerala?"
   - **Location:** Kerala, India
   - **Request:** Top 3 things to do
   - **Tone/Role:** Helpful and enthusiastic travel assistant

2.  **Identify Key Attractions/Experiences in Kerala:**
   Kerala is known for:
   - Backwaters (houseboat cruises, especially in Alleppey/Alleppey)
   - Hill stations & nature (Munnar tea plantations, wildlife, trekking)
   - Culture & wellness (Ayurveda, Kathakali dance, temples, beaches)
   - Wildlife (Periyar Tiger Reserve)
   - Food (Kerala cuisine, spice markets)

3.  **Select Top 3 (Balanced, Iconic, Diverse):**
   I need to pick 3 that represent the best of Kerala and offer different experiences:
   1. **Cruise the Backwaters on a Houseboat** (Iconic, unique to Kerala, relaxing)
   2. **Explore Munnar’s Tea Plantations & Hill Stations** (Nature, scenic, cultural)
   3. **Experience Ayurveda & Traditio

In [ ]:
# ── INSPECT EACH STEP SEPARATELY ────────────────────────────
# This makes the chain's internals visible

# What does the prompt look like after filling the template?
filled = prompt.invoke({'input': 'Top places in Kerala'})
print('📝 Filled Prompt (what the LLM actually receives):',
      str(filled)[:200], '...')
print()

# What does the raw LLM response look like before parsing?
raw = (prompt | llm).invoke({'input': 'Top places in Kerala'})
print('📦 Raw LLM response type:', raw.__class__.__name__)
print('   .content:', raw.content[:120], '...')

📝 Filled Prompt (what the LLM actually receives): messages=[SystemMessage(content='You are a helpful and enthusiastic travel assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Top places in Kerala', additional_kwargs={},  ...

📦 Raw LLM response type: AIMessage
   .content: 
<think>
Here's a thinking process:

1.  **Understand User Request:** The user is asking for "Top places in Kerala". Thi ...


In [ ]:
# ── PARAMETERISED PROMPT TEMPLATE ───────────────────────────

trip_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are an expert travel planner. '
     'Create a concise {days}-day itinerary for {destination} '
     'with a daily budget of {daily_budget} INR. '
     'Include must-visit places, local food, and practical tips.'),
    ('human', 'Plan my trip!')
])

trip_chain = trip_prompt | llm | StrOutputParser()
print('✅ Parameterised chain ready!')

✅ Parameterised chain ready!


In [ ]:
# ── SAME CHAIN, DIFFERENT DESTINATIONS ──────────────────────

print('=' * 55)
print('🏖️  GOA — 3 Days, ₹5,000/day')
print('=' * 55)
print(trip_chain.invoke({'destination': 'Goa', 'days': '3', 'daily_budget': '5000'}))

print()
print('=' * 55)
print('🏔️  MANALI — 4 Days, ₹4,000/day')
print('=' * 55)
print(trip_chain.invoke({'destination': 'Manali', 'days': '4', 'daily_budget': '4000'}))

🏖️  GOA — 3 Days, ₹5,000/day

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Role:** Expert travel planner
   - **Destination:** Goa
   - **Duration:** 3 days
   - **Budget:** 5000 INR per day (total 15,000 INR)
   - **Requirements:** Must-visit places, local food, practical tips
   - **Format:** Concise itinerary

2.  **Deconstruct Requirements:**
   - **Daily Budget:** 5000 INR/day is quite comfortable for Goa if managed well. Covers accommodation, food, transport, activities, and miscellaneous.
   - **Duration:** 3 days/2 nights (typical for a short trip)
   - **Must-visit places:** Need a mix of beaches, heritage, culture, and nature. South Goa for tranquility, North Goa for vibe/heritage.
   - **Local food:** Seafood, Goan cuisine (fish curry rice, vindaloo, bebinca, feni, etc.)
   - **Practical tips:** Transport, booking, safety, money, best time, etc.
   - **Concise:** Keep it structured, bullet-point friendly, no fluff.

3.  **Budget Breakdown (per day ~

In [ ]:
@tool
def my_tool(input: str) -> str:
    """LLM reads this docstring to understand what the tool does."""
    return result


In [ ]:
# ── TOOL 1: WEB SEARCH ──────────────────────────────────────
# Keep the docstring SHORT — the LLM reads it to decide when to call the tool.
# Long docstrings can confuse the function-call generator on Groq.

@tool
def web_search(query: str) -> str:
    '''Search the web for travel destination info, attractions, and tips.
    Args: query — search string.
    Returns: top search results as text.
    '''
    try:
        results = []
        with DDGS() as ddgs:
            for r in list(ddgs.text(query, max_results=3)):
                results.append(f"- {r.get('title','')}\n  {r.get('body','')}")
        # Truncate to 1500 chars — avoids overflowing the context window
        output = '\n\n'.join(results)
        return output[:1500] + '...' if len(output) > 1500 else output
    except Exception as e:
        return f'Search unavailable: {str(e)}'

print(f'✅ Tool 1: {web_search.name}')

✅ Tool 1: web_search


In [ ]:
# ── TEST WEB SEARCH STANDALONE ──────────────────────────────

result = web_search.invoke({'query': 'best places to visit in Rajasthan India travel guide'})
print('🔍 Search Result (first 500 chars):')
print(result[:500], '...')

/tmp/ipykernel_1170/596202386.py:13: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


🔍 Search Result (first 500 chars):
 ...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── TOOL 2: REAL-TIME WEATHER ───────────────────────────────

@tool
def get_weather(city: str) -> str:
    '''Get current weather for a city.
    Args: city — city name (e.g. Mumbai, Goa, Shimla).
    Returns: temperature, conditions, humidity, packing tip.
    '''
    try:
        resp = requests.get(
            'http://api.openweathermap.org/data/2.5/weather',
            params={'q': city, 'appid': OPENWEATHER_API_KEY, 'units': 'metric'},
            timeout=8
        )
        d = resp.json()
        if d.get('cod') == 200:
            cond  = d['weather'][0]['description'].capitalize()
            temp  = d['main']['temp']
            feels = d['main']['feels_like']
            hum   = d['main']['humidity']
            tip   = ('Pack light clothes!' if temp > 30 else
                     'Light layers recommended.' if temp > 20 else
                     'Carry a jacket.' if temp > 10 else
                     'Pack warm clothes!')
            return (f'Weather in {city}: {cond}, {temp}°C '
                    f'(feels {feels}°C), humidity {hum}%. Tip: {tip}')
        return f'Could not get weather for {city}. Check the city name.'
    except Exception as e:
        return f'Weather error: {str(e)}'

print(f'✅ Tool 2: {get_weather.name}')

✅ Tool 2: get_weather


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
result = get_weather.invoke({'city': 'Mumbai'})
print('🌤️ Current Weather:')
print(result)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🌤️ Current Weather:
Weather in Mumbai: Light rain, 29.99°C (feels 36.99°C), humidity 79%. Tip: Light layers recommended.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
import re, json, warnings
warnings.filterwarnings('ignore')

def _parse_tool_call(text):
    m = re.search(r'<function=(\w+)(\{.*?\})\s*(?:</function>)?', text, re.DOTALL)
    if m:
        try: return m.group(1), json.loads(m.group(2))
        except: return m.group(1), {}
    return None, None

def run_agent(user_input, tools, chat_history=None, verbose=True, max_iter=6):
    if chat_history is None: chat_history = []
    tools_map = {t.name: t for t in tools}
    tool_lines = '\n'.join([f'  {t.name}: {t.description.strip().split(chr(10))[0]}' for t in tools])
    system = (f"You are a travel assistant with these tools:\n{tool_lines}\n\n"
              "Call tools as: <function=tool_name{\"param\": \"value\"}></function>\n"
              "When done, write: Final Answer: <your answer>")
    messages = [SystemMessage(content=system)] + list(chat_history) + [HumanMessage(content=user_input)]
    for i in range(max_iter):
        if verbose: print(f"\n🔄 Step {i+1}")
        response = llm.invoke(messages)
        resp = response.content
        messages.append(AIMessage(content=resp))
        if verbose: print(f"💭 {resp[:200]}")
        m = re.search(r'(?i)final\s*answer\s*:\s*(.*)', resp, re.DOTALL)
        if m: return m.group(1).strip(), messages
        name, args = _parse_tool_call(resp)
        if name and name in tools_map:
            if verbose: print(f"🔧 {name}({args})")
            result = str(tools_map[name].invoke(args))[:700]
            if verbose: print(f"📥 {result[:200]}")
            messages.append(HumanMessage(content=f"Tool result ({name}):\n{result}\n\nContinue reasoning."))
        else:
            return resp, messages
    return "Max iterations reached.", messages

print("✅ run_agent() ready")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ run_agent() ready


In [ ]:
# ── AGENT SYSTEM MESSAGE ─────────────────────────────────────
# Instead of a ChatPromptTemplate for the agent, we just define the
# system message as a plain string. Our manual loop inserts it as a
# SystemMessage at the top of every call.

TRAVEL_SYSTEM_MSG = (
    'You are a travel planning assistant. '
    'Use the provided tools to answer questions with real-time data. '
    'Always call a tool when you need current or factual information.'
)

print('✅ System message defined.')
print('   Preview:', TRAVEL_SYSTEM_MSG[:80], '...')

✅ System message defined.
   Preview: You are a travel planning assistant. Use the provided tools to answer questions  ...


In [ ]:
# ── THE AGENT LOOP ───────────────────────────────────────────
# This replaces create_tool_calling_agent + AgentExecutor.
# Advantages: no streaming issues, fully transparent, runs on any Groq model.
#
# How it works:
#   1. Bind tools to the LLM so it knows what functions are available
#   2. Call llm_with_tools.invoke(messages)
#   3. If the LLM returns tool_calls → run each tool → append results → repeat
#   4. If no tool_calls → the LLM has a final answer → return it

def run_agent(user_input, tools, chat_history=None, verbose=True, max_iter=5):
    '''
    Manual ReAct-style agent loop using llm.bind_tools().
    Args:
        user_input  : the user question
        tools       : list of @tool-decorated functions
        chat_history: list of previous HumanMessage / AIMessage objects
        verbose     : print tool calls and results
        max_iter    : safety limit on iterations
    Returns:
        (final_answer_str, updated_messages_list)
    '''
    if chat_history is None:
        chat_history = []

    # Map tool names to callables
    tools_map = {t.name: t for t in tools}

    # Build the LLM with tools bound
    llm_with_tools = llm.bind_tools(tools)

    # Assemble starting messages
    messages = (
        [SystemMessage(content=TRAVEL_SYSTEM_MSG)]
        + list(chat_history)
        + [HumanMessage(content=user_input)]
    )

    for iteration in range(max_iter):
        if verbose:
            print(f'\n🔄 Step {iteration + 1}')

        response = llm_with_tools.invoke(messages)   # ← direct invoke, no streaming
        messages.append(response)

        # No tool calls → model has the final answer
        if not response.tool_calls:
            if verbose:
                print('✅ Done — no more tools needed.')
            return response.content, messages

        # Execute every tool the model requested
        for tc in response.tool_calls:
            name = tc['name']
            args = tc['args']
            if verbose:
                print(f'🔧 Tool called : {name}')
                print(f'   Arguments   : {args}')

            result = tools_map[name].invoke(args)

            if verbose:
                preview = str(result)[:200]
                print(f'📥 Tool result : {preview}...' if len(str(result)) > 200 else f'📥 Tool result : {result}')

            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tc['id']
            ))

    return 'Max iterations reached without a final answer.', messages

print('✅ run_agent() defined — manual tool-calling loop ready.')
print('   Uses: llm.bind_tools()  +  llm_with_tools.invoke()  (no streaming)')

✅ run_agent() defined — manual tool-calling loop ready.
   Uses: llm.bind_tools()  +  llm_with_tools.invoke()  (no streaming)


In [ ]:
# ── TEST QUERY 1: Destination Search ────────────────────────
# Watch the agent call web_search automatically

tools_s1 = [web_search, get_weather]

print('=' * 60)
print('Query 1: Best places in Goa')
print('=' * 60)

answer, _ = run_agent(
    user_input='What are the best places to visit in Goa and when is the ideal time to go?',
    tools=tools_s1,
    verbose=True
)
print()
print('━' * 60)
print('🤖 Final Answer:')
print('━' * 60)
print(answer)

Query 1: Best places in Goa

🔄 Step 1
🔧 Tool called : web_search
   Arguments   : {'query': 'best places to visit in Goa and ideal time to go'}


/tmp/ipykernel_1170/596202386.py:13: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


📥 Tool result : - Best Time to Visit Goa - Ideal Seasons & Travel Tips for 2026
  Nov 30, 2025 · Discover the best time to visit Goa, Know the weather, crowd level, prices, festivals, and ideal season for beaches, …
...

🔄 Step 2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔧 Tool called : web_search
   Arguments   : {'query': 'best places to visit in Goa'}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

📥 Tool result : - 52 Best Places to visit in Goa | Top Attractions & Sightseeing
  See most popular tourist places to visit in Goa, top things to do, shopping and nightlife in Goa, find entry timings, fees about vari...

🔄 Step 3


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Done — no more tools needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 Final Answer:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Goa is a vibrant destination known for its stunning beaches, rich history, and lively nightlife. Here is a guide to the best places to visit and the ideal time to plan your trip.

### **Ideal Time to Visit Goa**

*   **Best Time (November to February):** This is the peak season. The weather is pleasant, sunny, and breezy, making it perfect for beach activities and sightseeing. Expect higher prices and larger crowds during this period.
*   **Shoulder Season (March to May):** The weather gets hotter and more humid. While it is still a good time to visit if you don't mind the heat, it is generally less crowded than the winter months.
*   **Monsoon Season (June to October):** Goa receives heavy rainfall during these months. While the landscape turns lush and green, many water sports are suspended, and some beaches may be r

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── TEST QUERY 2: Weather Check ─────────────────────────────

print('=' * 60)
print('Query 2: Current weather in Shimla')
print('=' * 60)

answer, _ = run_agent(
    user_input='Check the weather in Shimla right now. Is it a good time to visit?',
    tools=tools_s1,
    verbose=True
)
print()
print('━' * 60)
print('🤖 Final Answer:')
print('━' * 60)
print(answer)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Query 2: Current weather in Shimla

🔄 Step 1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔧 Tool called : get_weather
   Arguments   : {'city': 'Shimla'}
📥 Tool result : Weather in Shimla: Light rain, 21.39°C (feels 21.88°C), humidity 88%. Tip: Light layers recommended.

🔄 Step 2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Done — no more tools needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 Final Answer:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The current weather in Shimla is **21.39°C** with **light rain** and high humidity (88%). It feels like 21.88°C.

**Is it a good time to visit?**
It is a pleasant temperature, but the light rain and high humidity might make outdoor sightseeing slightly damp. It is generally a good time to visit if you don't mind a bit of rain, as the temperature is comfortable.

**Packing Tip:**
Bring light layers and an umbrella or raincoat to stay comfortable.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── DEMONSTRATE THE FORGETTING PROBLEM ──────────────────────

print('🔴 WITHOUT MEMORY — Agent forgets between calls')
print('=' * 60)

# Turn 1: give context
print('Turn 1 — User gives context:')
ans1, _ = run_agent(
    'I want a 4-day trip to Coorg with a budget of 12000 INR.',
    tools=tools_s1, verbose=False
)
print('Agent:', ans1[:200], '...\n')

# Turn 2: no memory passed → agent knows nothing from Turn 1
print('Turn 2 — Follow-up (no history passed):')
ans2, _ = run_agent(
    'What if I extend by 2 more days? Will my budget hold?',
    tools=tools_s1, verbose=False   # ← no chat_history passed
)
print('Agent (forgot context!):', ans2[:200], '...')
print()
print('⚠️  The agent in Turn 2 has no idea about Coorg or the 12000 INR budget.')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔴 WITHOUT MEMORY — Agent forgets between calls
Turn 1 — User gives context:


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

Agent: Here is a budget-friendly 4-day itinerary for Coorg (Kodagu) tailored to a budget of **₹12,000 INR** (assuming travel for 2 people, or a solo traveler with slightly more comfort). This plan focuses on ...

Turn 2 — Follow-up (no history passed):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Agent (forgot context!): To help you figure out if your budget will hold, I'll need a few quick details:

1. **Destination**: Where are you traveling?
2. **Current Budget**: What's your total budget for the trip?
3. **Origina ...

⚠️  The agent in Turn 2 has no idea about Coorg or the 12000 INR budget.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag